In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.etl.context import PipelineContext
from src.etl.run_tracker import PipelineRun
from src.etl import extract, transform

13:17:23 | INFO    | etl          | Logging to: C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\logs\etl_20260511_131723.log


In [2]:
print("=" * 60)
print("Test 1: Sample mode — extract + transform")
print("=" * 60)

ctx = PipelineContext(run_mode="sample", sample_size=10_000)

with PipelineRun(ctx) as run:
    with run.stage("extract") as stage:
        extract.run(ctx)
        stage.set_metrics(rows_in=0, rows_out=ctx.rows_extracted)

    with run.stage("transform") as stage:
        df = transform.run(ctx)
        stage.set_metrics(rows_in=ctx.rows_extracted, rows_out=ctx.rows_transformed)

print(f"\n✅ Transformed {ctx.rows_transformed:,} rows × {len(df.columns)} cols")
print(f"\nDerived columns preview:")
print(df[['event_time', 'date_sk', 'time_sk', 'is_after_hours', 'is_weekend',
          'src_ip', 'src_ip_class', 'dest_ip', 'dest_ip_class',
          'src_port', 'dest_port', 'protocol_num',
          'attack_label', 'attack_family_denorm', 'is_attack']].head(5).to_string())

Test 1: Sample mode — extract + transform
13:17:46 | INFO    | etl.tracker  | ╔══ Pipeline run 9 started: cicids_to_warehouse (sample) ══╗
13:17:46 | INFO    | etl.tracker  | ▶ Starting stage: extract
13:17:46 | INFO    | etl.extract  | Reading stratified sample (n=10,000) from cicids_clean.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

13:18:07 | INFO    | etl.extract  | Loaded 8,047 rows × 87 columns into memory
13:18:07 | WARNING | etl.extract  | Dropped 1,000 invalid rows (missing timestamp or IPs)
13:18:07 | INFO    | etl.extract  | Attack family distribution (top): {'Benign': 1000, 'DDoS': 1000, 'DoS': 1000, 'Web Attack': 1000, 'Reconnaissance': 1000, 'Brute Force': 1000, 'Botnet': 1000, 'Infiltration': 36, 'Exploit': 11}
13:18:07 | INFO    | etl.tracker  | ✓ Stage 'extract' completed in 20.4s (rows: 0 → 7,047)
13:18:07 | INFO    | etl.tracker  | ▶ Starting stage: transform
13:18:07 | INFO    | etl.transform | Transforming 7,047 rows
13:18:07 | INFO    | etl.transform | Deriving date_sk, time_sk, is_after_hours, is_weekend
13:18:07 | INFO    | etl.transform | Classifying source IPs
13:18:07 | INFO    | etl.transform | Classifying destination IPs
13:18:07 | INFO    | etl.transform | Source IP classification: {'internal': 6549, 'external': 498}
13:18:07 | INFO    | etl.transform | Destination IP classification: {'

In [3]:
print("=" * 60)
print("Test 2: Date/time key validation")
print("=" * 60)

import pandas as pd

# date_sk distribution should match July 3-7, 2017
print("\ndate_sk distribution:")
print(df['date_sk'].value_counts().sort_index())

# Cross-check: derive expected date_sk from event_time and compare
expected = (df['event_time'].dt.year * 10000 +
            df['event_time'].dt.month * 100 +
            df['event_time'].dt.day)
mismatch = (df['date_sk'] != expected).sum()
print(f"\nDate SK mismatches: {mismatch}  (expected 0)")

# time_sk should be 0-1439
print(f"\ntime_sk range: {df['time_sk'].min()} to {df['time_sk'].max()}  (expected 0-1439)")

# is_after_hours sanity — events at 8am-6pm should NOT be after-hours
mid_business = df[(df['event_time'].dt.hour >= 8) & (df['event_time'].dt.hour <= 17)]
unexpected_after_hours = mid_business['is_after_hours'].sum()
print(f"\nBusiness-hours events incorrectly flagged after-hours: {unexpected_after_hours}  (expected 0)")

Test 2: Date/time key validation

date_sk distribution:
date_sk
20170703     246
20170704    1186
20170705    1209
20170706    1227
20170707    3179
Name: count, dtype: int64

Date SK mismatches: 0  (expected 0)

time_sk range: 522 to 1022  (expected 0-1439)

Business-hours events incorrectly flagged after-hours: 0  (expected 0)


In [4]:
print("=" * 60)
print("Test 3: IP classification cross-check")
print("=" * 60)

# Cross-check known IPs from Week 1 findings
known_ips = {
    '192.168.10.50': 'internal',     # web server target
    '192.168.10.8': 'internal',      # workstation
    '172.16.0.1': 'internal',        # DDoS source (RFC1918)
    '205.174.165.73': 'external',    # documented attacker
    '8.8.8.8': None,                 # not in data, but if found should be external
}

for ip, expected in known_ips.items():
    if expected is None:
        continue
    matches = df[df['src_ip'] == ip]
    if len(matches) == 0:
        matches = df[df['dest_ip'] == ip]
        actual_col = 'dest_ip_class'
    else:
        actual_col = 'src_ip_class'

    if len(matches) > 0:
        actual = matches[actual_col].iloc[0]
        symbol = "✅" if actual == expected else "❌"
        print(f"  {symbol} {ip}: expected={expected}, actual={actual}")
    else:
        print(f"  ⏭️ {ip}: not in sample (skipping)")

Test 3: IP classification cross-check
  ✅ 192.168.10.50: expected=internal, actual=internal
  ✅ 192.168.10.8: expected=internal, actual=internal
  ✅ 172.16.0.1: expected=internal, actual=internal
  ✅ 205.174.165.73: expected=external, actual=external


In [5]:
print("=" * 60)
print("Test 4: Full mode — extract + transform (~3M rows)")
print("=" * 60)

ctx_full = PipelineContext(run_mode="full")

with PipelineRun(ctx_full) as run:
    with run.stage("extract") as stage:
        extract.run(ctx_full)
        stage.set_metrics(rows_in=0, rows_out=ctx_full.rows_extracted)

    with run.stage("transform") as stage:
        df_full = transform.run(ctx_full)
        stage.set_metrics(rows_in=ctx_full.rows_extracted, rows_out=ctx_full.rows_transformed)

print(f"\n✅ Transformed {ctx_full.rows_transformed:,} rows")
print(f"\nFinal memory usage: {df_full.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nFull date_sk distribution:")
print(df_full['date_sk'].value_counts().sort_index())
print(f"\nFull IP classification breakdown:")
print(f"  Source IPs:\n{df_full['src_ip_class'].value_counts()}")
print(f"  Dest IPs:\n{df_full['dest_ip_class'].value_counts()}")

Test 4: Full mode — extract + transform (~3M rows)
13:20:00 | INFO    | etl.tracker  | ╔══ Pipeline run 10 started: cicids_to_warehouse (full) ══╗
13:20:00 | INFO    | etl.tracker  | ▶ Starting stage: extract
13:20:00 | INFO    | etl.extract  | Reading cicids_clean.parquet (293.0 MB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

13:20:31 | INFO    | etl.extract  | Loaded 3,119,345 rows × 87 columns into memory
13:20:41 | WARNING | etl.extract  | Dropped 288,602 invalid rows (missing timestamp or IPs)
13:20:42 | INFO    | etl.extract  | Attack family distribution (top): {'Benign': 2273097, 'DoS': 252661, 'Reconnaissance': 158930, 'DDoS': 128027, 'Brute Force': 13835, 'Web Attack': 2180, 'Botnet': 1966, 'Infiltration': 36, 'Exploit': 11}
13:20:42 | INFO    | etl.tracker  | ✓ Stage 'extract' completed in 42.1s (rows: 0 → 2,830,743)
13:20:42 | INFO    | etl.tracker  | ▶ Starting stage: transform
13:20:42 | INFO    | etl.transform | Transforming 2,830,743 rows
13:20:44 | INFO    | etl.transform | Deriving date_sk, time_sk, is_after_hours, is_weekend
13:20:46 | INFO    | etl.transform | Classifying source IPs
13:20:51 | INFO    | etl.transform | Classifying destination IPs
13:20:54 | INFO    | etl.transform | Source IP classification: {'internal': 2467175, 'external': 363568}
13:20:54 | INFO    | etl.transform | Des